<a href="https://colab.research.google.com/github/rudalshan0412-code/Intent_Classifier-RAG_Chatbot/blob/main/06)_%EB%AA%A8%EB%93%88_%EC%A1%B0%EB%A6%BD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 구글 드라이브 연결

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# 프로젝트 루트로 이동

%cd /content/drive/MyDrive/rag_intent_chatbot

# 현재 경로 확인
import os

print("현재 작업 경로:", os.getcwd())

/content/drive/MyDrive/rag_intent_chatbot
현재 작업 경로: /content/drive/MyDrive/rag_intent_chatbot


In [ ]:
# 필수 파일 확인

from pathlib import Path

required_files = [
    "models/intent_classifier.pt",
    "data/intents.json",
    "data/documents/sample.txt",
    "src/intent/predict.py",
    "src/rag/document_loader.py",
    "src/rag/chunker.py",
    "src/rag/embedder.py",
    "src/rag/vector_store.py",
    "src/rag/retriever.py",
    "src/chatbot.py",
]

print("필수 파일 확인")
print("-" * 60)

for file_path in required_files:
    path = Path(file_path)
    status = "[있음]" if path.exists() else "[없음]"
    print(f"{status} {file_path}")

필수 파일 확인
------------------------------------------------------------
[있음] models/intent_classifier.pt
[있음] data/intents.json
[있음] data/documents/sample.txt
[있음] src/intent/predict.py
[있음] src/rag/document_loader.py
[있음] src/rag/chunker.py
[있음] src/rag/embedder.py
[있음] src/rag/vector_store.py
[있음] src/rag/retriever.py
[있음] src/chatbot.py


In [ ]:
# main.py

%%writefile main.py
"""RAG 챗봇과 Intent Classifier를 통합 실행한다."""

from pathlib import Path

from src.intent.predict import IntentPredictor
from src.rag.document_loader import Document, load_documents
from src.rag.text_preprocessor import preprocess_text
from src.rag.chunker import chunk_documents
from src.rag.embedder import TextEmbedder
from src.rag.vector_store import VectorStore
from src.rag.retriever import Retriever
from src.chatbot import Chatbot, ChatbotResult


MODEL_PATH = "models/intent_classifier.pt"
DOCUMENT_DIRECTORY = "data/documents"

CONFIDENCE_THRESHOLD = 0.60
RETRIEVAL_TOP_K = 3

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
PREVIEW_LENGTH = 200

EXIT_COMMANDS = {"exit", "quit", "종료"}


def build_retriever() -> Retriever:
    """문서를 처리하고 검색에 사용할 Retriever를 생성한다."""

    document_directory = Path(DOCUMENT_DIRECTORY) # 경로파일로 변경

    if not document_directory.exists():
        raise FileNotFoundError(
            f"문서 디렉터리를 찾을 수 없습니다: {DOCUMENT_DIRECTORY}"
        )

    text_files = list(document_directory.glob("*.txt")) # txt로 존재하는 모든 파일 불러오기

    if not text_files:
        raise FileNotFoundError(
            f"불러올 .txt 문서가 없습니다: {DOCUMENT_DIRECTORY}"
        )

    print("[1/6] 문서를 불러오는 중...")

    documents = load_documents( # 하위 txt 파일들의 모든 문서 내용을 불러오는 코드
        directory_path=DOCUMENT_DIRECTORY,
    )

    if not documents:
        raise RuntimeError("문서를 불러왔지만 문서 목록이 비어 있습니다.")

    print(f"      불러온 문서 수: {len(documents)}")

    print("[2/6] 문서를 전처리하는 중...")

    preprocessed_documents: list[Document] = []

    for document in documents:
        preprocessed_document = Document( # 문서의 내용과 출처, 정보를 함께 저장
            text=preprocess_text(document.text), # 청킹 전 전처리(BOM 제거)
            metadata=document.metadata,
        )

        preprocessed_documents.append(preprocessed_document)

    print("[3/6] 문서를 Chunk로 나누는 중...")

    chunks = chunk_documents( # 문서를 청킹
        documents=preprocessed_documents,
        chunk_size=CHUNK_SIZE,
        chunk_overlap=CHUNK_OVERLAP,
    )

    if not chunks:
        raise RuntimeError("문서에서 Chunk가 생성되지 않았습니다.")

    print(f"      생성된 Chunk 수: {len(chunks)}")

    print("[4/6] TextEmbedder를 불러오는 중...")

    try:
        embedder = TextEmbedder()
    except Exception as error:
        raise RuntimeError(
            "Sentence Transformer 모델을 불러오지 못했습니다."
        ) from error

    print("[5/6] Chunk 임베딩을 생성하는 중...")

    chunk_texts = [
        chunk.text #document_loader.py 에 있는 .text
        for chunk in chunks
    ]

    chunk_embeddings = embedder.encode_texts( # 2차원 임베딩 배열로 변환
        texts=chunk_texts,
    )

    print("[6/6] VectorStore와 Retriever를 생성하는 중...")

    try:
        vector_store = VectorStore() # 청크와 임배딩을 저장 및 검색

        vector_store.add(
            chunks=chunks,
            embeddings=chunk_embeddings,
        )
    except Exception as error:
        raise RuntimeError(
            "VectorStore 생성 또는 데이터 저장에 실패했습니다."
        ) from error

    retriever = Retriever(
        embedder=embedder,
        vector_store=vector_store,
    )

    print("      Retriever 생성 완료")

    return retriever


def build_chatbot() -> Chatbot:
    """IntentPredictor와 Retriever를 이용해 Chatbot을 생성한다."""

    model_path = Path(MODEL_PATH)

    if not model_path.exists():
        raise FileNotFoundError(
            f"학습된 Intent 모델을 찾을 수 없습니다: {MODEL_PATH}"
        )

    print("IntentPredictor를 불러오는 중...")

    intent_predictor = IntentPredictor(
        model_path=MODEL_PATH,
    )

    retriever = build_retriever()

    chatbot = Chatbot(
        intent_predictor=intent_predictor,
        retriever=retriever,
        confidence_threshold=CONFIDENCE_THRESHOLD,
        retrieval_top_k=RETRIEVAL_TOP_K,
        preview_length=PREVIEW_LENGTH,
    )

    return chatbot


def print_result(result: ChatbotResult) -> None:
    """Chatbot 처리 결과를 읽기 쉬운 형태로 출력한다."""

    print()
    print("-" * 70)
    print(f"예측 Intent : {result.predicted_intent}")
    print(f"Confidence  : {result.confidence:.4f}")
    print(f"Fallback    : {result.is_fallback}")
    print(f"RAG 사용    : {result.requires_rag}")
    print(f"검색 결과 수: {len(result.search_results)}")
    print()
    print(f"챗봇: {result.response}")

    if result.search_results:
        print()
        print("[문서 검색 결과]")

        for search_result in result.search_results:
            chunk = search_result.chunk

            preview = chunk.text[:PREVIEW_LENGTH]

            if len(chunk.text) > PREVIEW_LENGTH:
                preview += "..."

            print()
            print(
                f"{search_result.rank}위 | "
                f"score={search_result.score:.4f}"
            )
            print(f"source   : {chunk.source}")
            print(f"chunk_id : {chunk.chunk_id}")
            print(f"text     : {preview}")

    print("-" * 70)


def main() -> None:
    """챗봇을 초기화하고 사용자 입력을 반복해서 처리한다."""

    print("=" * 70)
    print("RAG 챗봇 + PyTorch Intent Classifier")
    print("=" * 70)

    try:
        chatbot = build_chatbot()

        print()
        print("챗봇 초기화가 완료되었습니다.")
        print("질문을 입력해주세요.")
        print("종료 명령: exit, quit, 종료")
        print()

        while True:
            user_input = input("사용자: ").strip()

            if user_input.lower() in EXIT_COMMANDS:
                print("챗봇을 종료합니다.")
                break

            if not user_input:
                print("챗봇: 질문을 입력해주세요.")
                continue

            result = chatbot.process_message(
                user_input=user_input,
            )

            print_result(result)

    except KeyboardInterrupt:
        print()
        print("사용자 요청으로 챗봇을 종료합니다.")

    except FileNotFoundError as error:
        print()
        print(f"[파일 오류] {error}")

    except RuntimeError as error:
        print()
        print(f"[초기화 오류] {error}")

    except Exception as error:
        print()
        print(f"[예상하지 못한 오류] {error}")


if __name__ == "__main__":
    main()

Overwriting main.py


In [ ]:
# 파일 내용 확인

!sed -n '1,320p' main.py

"""RAG 챗봇과 Intent Classifier를 통합 실행한다."""

from pathlib import Path

from src.intent.predict import IntentPredictor
from src.rag.document_loader import Document, load_documents
from src.rag.text_preprocessor import preprocess_text
from src.rag.chunker import chunk_documents
from src.rag.embedder import TextEmbedder
from src.rag.vector_store import VectorStore
from src.rag.retriever import Retriever
from src.chatbot import Chatbot, ChatbotResult


MODEL_PATH = "models/intent_classifier.pt"
DOCUMENT_DIRECTORY = "data/documents"

CONFIDENCE_THRESHOLD = 0.60
RETRIEVAL_TOP_K = 3

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
PREVIEW_LENGTH = 200

EXIT_COMMANDS = {"exit", "quit", "종료"}


def build_retriever() -> Retriever:
    """문서를 처리하고 검색에 사용할 Retriever를 생성한다."""

    document_directory = Path(DOCUMENT_DIRECTORY) # 경로파일로 변경

    if not document_directory.exists():
        raise FileNotFoundError(
            f"문서 디렉터리를 찾을 수 없습니다: {DOCUMENT_DIRECTORY}"
        )

    text_files = list(document_dire

In [ ]:
# build_retriever 테스트
from main import build_retriever

retriever = build_retriever()

print()
print("Retriever 타입:", type(retriever))
print("Retriever 초기화 성공")

[1/6] 문서를 불러오는 중...
      불러온 문서 수: 1
[2/6] 문서를 전처리하는 중...
[3/6] 문서를 Chunk로 나누는 중...
      생성된 Chunk 수: 9
[4/6] TextEmbedder를 불러오는 중...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

/content/drive/MyDrive/rag_intent_chatbot/src/rag/embedder.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.model.get_sentence_embedding_dimension() # 출력하는 벡터(임베딩)의 차원 수(크기)를 반환


[5/6] Chunk 임베딩을 생성하는 중...
[6/6] VectorStore와 Retriever를 생성하는 중...
      Retriever 생성 완료

Retriever 타입: <class 'src.rag.retriever.Retriever'>
Retriever 초기화 성공


In [ ]:
# build_chatbot 테스트
from main import build_chatbot

chatbot = build_chatbot()

print()
print("Chatbot 타입:", type(chatbot))
print("Chatbot 초기화 성공")

IntentPredictor를 불러오는 중...
[1/6] 문서를 불러오는 중...
      불러온 문서 수: 1
[2/6] 문서를 전처리하는 중...
[3/6] 문서를 Chunk로 나누는 중...
      생성된 Chunk 수: 9
[4/6] TextEmbedder를 불러오는 중...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

/content/drive/MyDrive/rag_intent_chatbot/src/rag/embedder.py:29: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dimension = self.model.get_sentence_embedding_dimension() # 출력하는 벡터(임베딩)의 차원 수(크기)를 반환


[5/6] Chunk 임베딩을 생성하는 중...
[6/6] VectorStore와 Retriever를 생성하는 중...
      Retriever 생성 완료


TypeError: Chatbot.__init__() missing 1 required positional argument: 'answer_generator'

In [ ]:
# 한문장 단위 통합 테스트

from main import build_chatbot, print_result

chatbot = build_chatbot()

test_inputs = [
    "안녕하세요",
    "고마워요",
    "너는 누구야?",
    "사용 방법을 알려줘",
    "문서에서 임베딩 내용을 찾아줘",
    "문서에서 Chunk를 사용하는 이유를 찾아줘",
    "파란 생각이 조용하게 숫자를 걸어간다",
]

for test_input in test_inputs:
    print()
    print("=" * 70)
    print(f"사용자 입력: {test_input}")

    result = chatbot.process_message(
        user_input=test_input,
    )

    print_result(result)

In [ ]:
# 전체 대화 반복문 실행
from main import main

main()
# 종료를 입력하면 프로그램 종료